# Étape 5 – Machine Learning : prédire les avis négatifs

**Objectif métier :** repérer **à l'avance** les commandes qui risquent de recevoir un avis négatif (1 ou 2 étoiles),
pour que le service client puisse agir (prévenir le client, accélérer l'envoi, offrir un geste commercial…).

**Type de problème :** c'est une **classification binaire** — le modèle doit répondre *oui* (avis négatif) ou *non*.

**Le principe du Machine Learning supervisé :**
1. on montre au modèle des milliers de commandes **dont on connaît déjà la réponse** (la note) → c'est l'**entraînement** ;
2. le modèle apprend tout seul les liens entre les caractéristiques de la commande et la note ;
3. on vérifie ensuite s'il devine juste sur des commandes **qu'il n'a jamais vues** → c'est le **test**.

> 💡 Vocabulaire : les caractéristiques (délai, prix, distance…) s'appellent les **features** ;
> ce qu'on veut prédire (avis négatif oui/non) s'appelle la **cible** (*target*).

## 0. Préparation

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import pg_url

engine = create_engine(pg_url())
COULEUR = "#2a78d6"
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

## 1. Construire le jeu de données : une ligne par commande

On rassemble en **une seule requête SQL** tout ce qu'on sait sur chaque commande livrée :
prix, frais de port, poids et volume des produits, catégorie, États du client et du vendeur,
coordonnées GPS, moyen de paiement, délais… et bien sûr la note.

*(La requête est longue mais tu connais déjà toutes les briques : CTE, `JOIN`, `GROUP BY`, `LEFT JOIN`.
Deux nouveautés : `DISTINCT ON` garde une seule ligne par commande — l'article le plus cher —
et `ARRAY_AGG(...)[1]` récupère le moyen de paiement du plus gros paiement.)*

In [2]:
REQUETE = """
    WITH notes AS (
        SELECT order_id, AVG(review_score) AS note
        FROM order_reviews
        GROUP BY order_id
    ),
    articles AS (
        SELECT i.order_id,
               COUNT(*)                                                         AS nb_articles,
               COUNT(DISTINCT i.seller_id)                                      AS nb_vendeurs,
               SUM(i.price)                                                     AS prix_total,
               SUM(i.freight_value)                                             AS frais_port,
               SUM(p.product_weight_g)                                          AS poids_total_g,
               SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS volume_total_cm3
        FROM order_items i
        JOIN products p ON p.product_id = i.product_id
        GROUP BY i.order_id
    ),
    article_principal AS (           -- l'article le plus cher de la commande
        SELECT DISTINCT ON (i.order_id)
               i.order_id, i.seller_id,
               COALESCE(t.product_category_name_english, 'inconnue') AS categorie
        FROM order_items i
        JOIN products p ON p.product_id = i.product_id
        LEFT JOIN category_translation t ON t.product_category_name = p.product_category_name
        ORDER BY i.order_id, i.price DESC
    ),
    paiements AS (
        SELECT order_id,
               MAX(payment_installments)                                AS nb_echeances,
               (ARRAY_AGG(payment_type ORDER BY payment_value DESC))[1] AS moyen_paiement
        FROM order_payments
        GROUP BY order_id
    )
    SELECT o.order_id,
           EXTRACT(MONTH FROM o.order_purchase_timestamp)::int                            AS mois_achat,
           o.order_estimated_delivery_date::date - o.order_purchase_timestamp::date       AS delai_promis_jours,
           a.nb_articles, a.nb_vendeurs, a.prix_total, a.frais_port,
           a.poids_total_g, a.volume_total_cm3,
           ap.categorie,
           c.customer_state AS etat_client,
           s.seller_state   AS etat_vendeur,
           gc.lat AS lat_client,  gc.lng AS lng_client,
           gs.lat AS lat_vendeur, gs.lng AS lng_vendeur,
           pm.moyen_paiement, pm.nb_echeances,
           o.order_delivered_customer_date::date - o.order_purchase_timestamp::date      AS delai_jours,
           o.order_delivered_customer_date::date - o.order_estimated_delivery_date::date AS ecart_jours,
           n.note
    FROM orders o
    JOIN notes             n  ON n.order_id  = o.order_id
    JOIN articles          a  ON a.order_id  = o.order_id
    JOIN article_principal ap ON ap.order_id = o.order_id
    JOIN customers         c  ON c.customer_id = o.customer_id
    JOIN sellers           s  ON s.seller_id   = ap.seller_id
    LEFT JOIN geolocation_zip gc ON gc.zip_code_prefix = c.customer_zip_code_prefix
    LEFT JOIN geolocation_zip gs ON gs.zip_code_prefix = s.seller_zip_code_prefix
    LEFT JOIN paiements       pm ON pm.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
"""

df = pd.read_sql(REQUETE, engine)
print(f"{len(df):,} commandes x {df.shape[1]} colonnes".replace(",", " "))
df.head()

95 824 commandes x 21 colonnes


,order_id,mois_achat,delai_promis_jours,nb_articles,nb_vendeurs,prix_total,frais_port,poids_total_g,volume_total_cm3,categorie,...,etat_vendeur,lat_client,lng_client,lat_vendeur,lng_vendeur,moyen_paiement,nb_echeances,delai_jours,ecart_jours,note
0,00018f77f2f0320c557190d7a144bdd3,4,19,1,1,239.90,19.93,30000.0,60000.0,pet_shop,...,SP,-20.205187,-50.927017,-23.565803,-46.519192,credit_card,3.0,16,-3,4.0
1,00010242fe8c5a6d1ba2dd792cb16214,9,16,1,1,58.90,13.29,650.0,3528.0,cool_stuff,...,SP,-21.763009,-41.310799,-22.499217,-44.125997,credit_card,2.0,7,-9,5.0
2,00024acbcdf0a6daa1e931b038114c75,8,12,1,1,12.99,12.79,200.0,2400.0,perfumery,...,SP,-23.106036,-46.592481,-20.554801,-47.387749,credit_card,2.0,6,-6,4.0
3,00048cc3ae777c65dbb7d2a0634bc1ea,5,22,1,1,21.90,12.69,450.0,2880.0,housewares,...,SP,-19.740232,-47.925038,-21.189678,-47.787260,boleto,1.0,7,-15,4.0
4,0005f50442cb953dcd1d21e1fb923495,7,21,1,1,53.99,11.40,850.0,1827.0,books_technical,...,SP,-23.556739,-46.899006,-23.467609,-46.586825,credit_card,1.0,2,-19,4.0


## 2. Créer de nouvelles features (« feature engineering »)

Les données brutes ne suffisent pas toujours : on **fabrique** des variables plus parlantes pour le modèle.
- **`distance_km`** : distance à vol d'oiseau entre le client et le vendeur (formule de **haversine**, qui tient compte de la courbure de la Terre)
- **`meme_etat`** : le vendeur est-il dans le même État que le client ? (1 = oui, 0 = non)
- **`avis_negatif`** : notre **cible** → 1 si la note est ≤ 2, sinon 0

In [3]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Distance en km entre deux points GPS."""
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(df["lat_client"], df["lng_client"], df["lat_vendeur"], df["lng_vendeur"])
df["meme_etat"] = (df["etat_client"] == df["etat_vendeur"]).astype(int)
df["avis_negatif"] = (df["note"] <= 2).astype(int)

df[["distance_km", "meme_etat", "avis_negatif"]].describe().round(2)

,distance_km,meme_etat,avis_negatif
count,95348.00,95824.00,95824.00
mean,600.09,0.36,0.13
std,592.01,0.48,0.33
min,0.00,0.00,0.00
25%,184.96,0.00,0.00
50%,433.77,0.00,0.00
75%,798.23,1.00,0.00
max,3398.55,1.00,1.00


## 3. La cible est-elle équilibrée ?

In [4]:
repartition = df["avis_negatif"].value_counts(normalize=True).mul(100).round(1)
repartition.index = ["Avis positif / neutre (0)", "Avis négatif (1)"]
repartition

Avis positif / neutre (0)    87.2
Avis négatif (1)             12.8
Name: proportion, dtype: float64

➡️ Les classes sont **déséquilibrées** : il y a beaucoup plus de clients contents que de mécontents.
On verra que c'est un **piège** pour évaluer un modèle.

## 4. ⚠️ La fuite de données (« data leakage »)

C'est **l'erreur n°1** en Machine Learning, et une question classique d'entretien.

Imagine qu'on donne au modèle `ecart_jours` (le retard réel). Il deviendrait très bon… mais c'est de la **triche** :
**au moment où l'on veut prédire** (juste après l'achat), le colis n'est pas encore livré, on **ne connaît pas encore** le retard !
Un modèle qui utilise une information du futur paraît excellent en test… et ne sert à rien en vrai.

On construit donc **deux scénarios** :

| Scénario | Quand prédit-on ? | Features autorisées |
|---|---|---|
| **A – À l'achat** | juste après la commande | tout ce qu'on sait à l'achat (prix, distance, délai promis…) |
| **B – À la livraison** | quand le colis arrive, **avant** que le client note | A + le délai réel et le retard |

Le scénario B est utile aussi : le service client peut contacter le client **entre la livraison et l'avis**.

In [5]:
FEATURES_NUM_ACHAT = ["delai_promis_jours", "nb_articles", "nb_vendeurs", "prix_total", "frais_port",
                      "poids_total_g", "volume_total_cm3", "distance_km", "meme_etat",
                      "nb_echeances", "mois_achat"]
FEATURES_CAT = ["categorie", "etat_client", "moyen_paiement"]
FEATURES_LIVRAISON = ["delai_jours", "ecart_jours"]

FEATURES_A = FEATURES_NUM_ACHAT + FEATURES_CAT                       # scénario A
FEATURES_B = FEATURES_NUM_ACHAT + FEATURES_LIVRAISON + FEATURES_CAT  # scénario B

print(f"Scénario A : {len(FEATURES_A)} features")
print(f"Scénario B : {len(FEATURES_B)} features")

Scénario A : 14 features
Scénario B : 16 features


## 5. Séparer entraînement et test

On coupe les données en deux :
- **80 % pour l'entraînement** : le modèle apprend sur ces commandes ;
- **20 % pour le test** : on les cache au modèle, puis on vérifie ses prédictions dessus.

C'est comme un examen : si on donnait à l'élève les questions de l'examen pendant ses révisions,
sa note ne mesurerait pas ce qu'il a vraiment compris.

`stratify=y` garde **la même proportion d'avis négatifs** dans les deux parties.
`random_state=42` rend le découpage **reproductible** (même résultat à chaque exécution).

In [6]:
from sklearn.model_selection import train_test_split

X = df[FEATURES_B]          # on garde toutes les colonnes ; chaque modèle choisira les siennes
y = df["avis_negatif"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Entraînement : {len(X_train):,} commandes".replace(",", " "))
print(f"Test         : {len(X_test):,} commandes".replace(",", " "))
print(f"% d'avis négatifs : train {100 * y_train.mean():.1f} % | test {100 * y_test.mean():.1f} %")

Entraînement : 76 659 commandes
Test         : 19 165 commandes
% d'avis négatifs : train 12.8 % | test 12.8 %


## 6. Le modèle « bête » de référence (baseline)

Avant de construire un vrai modèle, on se demande : **quel score obtiendrait un modèle qui ne réfléchit pas ?**

Le plus simple : répondre **toujours « avis positif »**. On mesure deux choses :
- **Accuracy (exactitude)** : % de prédictions correctes, toutes classes confondues ;
- **Rappel (recall) des avis négatifs** : parmi les **vrais** avis négatifs, combien le modèle en a-t-il **détectés** ?

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print(f"Accuracy                   : {100 * accuracy_score(y_test, pred):.1f} %")
print(f"Rappel des avis négatifs   : {100 * recall_score(y_test, pred):.1f} %")

Accuracy                   : 87.2 %
Rappel des avis négatifs   : 0.0 %


### 🎯 À toi d'interpréter

1. L'accuracy du modèle « bête » est-elle **bonne ou mauvaise** à première vue ? Pourquoi est-elle si élevée ?
2. Que vaut son **rappel** ? Ce modèle est-il **utile** pour le service client ?
3. Si un collègue t'annonce fièrement « mon modèle a 85 % d'accuracy ! », que lui réponds-tu ?
4. Pour chacune de ces features, dis si elle est autorisée dans le **scénario A** (à l'achat) : `frais_port`, `ecart_jours`, `distance_km`, `note`.

✍️ *Écris tes réponses ici :*

1. 87,2 % : élevée mais trompeuse. Le modèle répond toujours "positif" et 87,2 % des commandes le sont : l'accuracy reflète seulement le déséquilibre des classes.
2. Rappel = 0 % : il ne détecte aucun client mécontent, donc il est inutile pour le service client.
3. 85 % d'accuracy, c'est moins bien que le modèle bête (87,2 %). Sur des classes déséquilibrées, il faut regarder le rappel et la précision, et toujours comparer à une baseline.
4. frais_port : oui. distance_km : oui. ecart_jours : non (connu seulement après la livraison = fuite de données). note : non (c'est la cible).

# Partie 2 – Les vrais modèles

## 7. Préparer les données pour le modèle : le *pipeline*

Un modèle ne comprend que des **nombres**, sans **valeurs manquantes**. Il faut donc préparer les données :
- **Valeurs manquantes** (`SimpleImputer`) : ex. `distance_km` est vide quand un code postal n'a pas de coordonnées → on la remplace par la **médiane** ;
- **Mise à l'échelle** (`StandardScaler`) : `prix_total` va jusqu'à 13 000 alors que `meme_etat` vaut 0 ou 1 → on ramène tout à la même échelle, sinon certains modèles donnent trop de poids aux grands nombres ;
- **Encodage des catégories** (`OneHotEncoder`) : la colonne `etat_client` = « SP », « RJ »… devient une colonne 0/1 par État.
  Les catégories rares (< 100 commandes) sont regroupées dans « autres ».

Un **`Pipeline`** enchaîne la préparation **et** le modèle en un seul objet.
Gros avantage : la préparation est **apprise uniquement sur le train**, puis appliquée au test → pas de fuite de données.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def construire_pipeline(features, modele):
    """Préparation des données + modèle, enchaînés dans un seul objet."""
    num = [f for f in features if f not in FEATURES_CAT]
    cat = [f for f in features if f in FEATURES_CAT]
    preparation = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100), cat),
    ])
    return Pipeline([("preparation", preparation), ("modele", modele)])

## 8. Comment juger un modèle : la matrice de confusion

Pour chaque commande du test, le modèle prédit « positif » ou « négatif ». On compare à la réalité :

| | **Prédit : positif** | **Prédit : négatif** |
|---|---|---|
| **Réel : positif** | ✅ Vrai négatif — client content, laissé tranquille | ⚠️ **Fausse alerte** — on contacte un client content pour rien |
| **Réel : négatif** | ❌ **Raté** — client mécontent non détecté | ✅ **Bonne détection** — client mécontent repéré à temps |

À partir de là, on calcule :
- **Rappel** = bonnes détections ÷ tous les vrais mécontents → *« quelle part des mécontents attrape-t-on ? »*
- **Précision** = bonnes détections ÷ toutes les alertes → *« quand le modèle alerte, a-t-il raison ? »*
- **F1** = compromis entre rappel et précision (moyenne harmonique)
- **ROC-AUC** = capacité à **classer** les commandes de la moins risquée à la plus risquée : **0,5 = hasard**, **1 = parfait**

💼 **Le choix dépend du métier** : si contacter un client coûte peu (un e-mail), on préfère un **bon rappel** quitte à avoir des fausses alertes.
Si chaque contact coûte cher (un geste commercial), on veut une **bonne précision**.

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

RESULTATS = []

def evaluer(nom, modele, features):
    """Entraîne le modèle, affiche sa matrice de confusion et enregistre ses scores."""
    pipeline = construire_pipeline(features, modele)
    pipeline.fit(X_train[features], y_train)
    pred = pipeline.predict(X_test[features])
    proba = pipeline.predict_proba(X_test[features])[:, 1]

    scores = {"modèle": nom,
              "accuracy": accuracy_score(y_test, pred),
              "précision": precision_score(y_test, pred, zero_division=0),
              "rappel": recall_score(y_test, pred),
              "F1": f1_score(y_test, pred),
              "ROC-AUC": roc_auc_score(y_test, proba)}
    RESULTATS.append(scores)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["Positif", "Négatif"],
        cmap="Blues", colorbar=False, values_format=",", ax=ax)
    ax.set_title(nom, loc="left", fontweight="bold")
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    plt.show()
    return pipeline, pd.Series(scores).drop("modèle").astype(float).round(3)

# On ajoute la baseline au tableau de comparaison
_, scores = evaluer("Baseline (toujours positif)", DummyClassifier(strategy="most_frequent"), FEATURES_A)
scores

## 9. Modèle 1 : la régression logistique

Malgré son nom, c'est un modèle de **classification**. Il donne un **poids** à chaque feature
(ex. « +1 jour de délai promis → risque un peu plus élevé ») et additionne ces poids pour calculer une **probabilité** d'avis négatif.
Simple, rapide, et **interprétable** : c'est le premier modèle à essayer.

⚖️ **`class_weight="balanced"`** : comme les mécontents ne sont que 12,8 %, le modèle aurait tendance à les ignorer
(comme la baseline !). Cette option leur donne **plus de poids** pendant l'apprentissage : rater un mécontent « coûte » plus cher.

**Scénario A – à l'achat :**

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_A, scores = evaluer("Logistique – A (achat)",
                           LogisticRegression(class_weight="balanced", max_iter=1000), FEATURES_A)
scores

## 10. Modèle 2 : l'arbre de décision

Un arbre pose une **suite de questions oui/non**, comme un jeu de « Qui est-ce ? » :
*« Le délai promis dépasse-t-il 25 jours ? → Oui → Le client est-il dans un autre État ? → … »*

`max_depth=6` limite l'arbre à 6 questions d'affilée : sans limite, il apprendrait **par cœur** les commandes d'entraînement
(c'est le **surapprentissage**, ou *overfitting*) et se tromperait sur les nouvelles.
`min_samples_leaf=50` impose au moins 50 commandes par décision finale, pour la même raison.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbre_A, scores = evaluer("Arbre – A (achat)",
                          DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                                 class_weight="balanced", random_state=42),
                          FEATURES_A)
scores

## 11. Scénario B : à la livraison (on ajoute le délai réel et le retard)

In [ ]:
logreg_B, scores = evaluer("Logistique – B (livraison)",
                           LogisticRegression(class_weight="balanced", max_iter=1000), FEATURES_B)
scores

In [ ]:
arbre_B, scores = evaluer("Arbre – B (livraison)",
                          DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                                 class_weight="balanced", random_state=42),
                          FEATURES_B)
scores

## 12. Tableau comparatif

In [ ]:
comparaison = pd.DataFrame(RESULTATS).set_index("modèle").round(3)
comparaison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
barres = ax.barh(comparaison.index[::-1], comparaison["ROC-AUC"][::-1], color=COULEUR, height=0.6)
ax.bar_label(barres, labels=[f"{v:.3f}" for v in comparaison["ROC-AUC"][::-1]], padding=4)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax.text(0.51, len(comparaison) - 0.45, "hasard", color="gray", fontsize=10, va="bottom")
ax.set_ylim(-0.6, len(comparaison) + 0.1)
ax.set_xlim(0, 1)
ax.set_title("ROC-AUC par modèle (0,5 = hasard, 1 = parfait)", loc="left", fontweight="bold")
plt.show()

### 🎯 À toi d'interpréter

1. Avec `class_weight="balanced"`, l'**accuracy** de la régression logistique A est-elle meilleure ou moins bonne que la baseline ? Est-ce grave ? Regarde son **rappel**.
2. Dans la matrice de confusion « Logistique – A » : combien de **mécontents détectés** ? Combien de **fausses alertes** ?
3. Compare la **ROC-AUC** du scénario A et du scénario B. Pourquoi un tel écart ?
4. Le scénario A (à l'achat) est-il **inutile** pour autant ? (Compare à 0,5.)
5. 💼 Tu es responsable du service client : quel modèle déploies-tu, et **à quel moment** contactes-tu le client ?

✍️ *Écris tes réponses ici :*

…